Very rough timings for 'jina-clip-v2' on my CPU (Apple M2):
- [Done] HF implementation from https://huggingface.co/jinaai/jina-clip-v2
- [Done] ONNX + int8 dynamic quantization

In [1]:
# %pip install onnxruntime transformers torch torchvision requests Pillow --no-cache-dir

In [2]:
import io
import statistics
import time

import numpy as np
import onnxruntime as ort
import requests
import torch
from huggingface_hub import hf_hub_download
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer

/opt/miniconda3/envs/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Auxiliary functions

In [3]:
def measure(fn: callable, warmup: int, runs: int) -> dict:
    """Measure the execution time of a function

    Args:
        fn (callable): The function to measure
        warmup (int): Number of warmup runs to perform before measuring
        runs (int): Number of runs to perform for measuring

    Returns:
        dict: A dictionary containing the mean, median, standard deviation, minimum, and maximum execution times in milliseconds
    """

    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)

    return {
        "mean": statistics.mean(times),
        "median": statistics.median(times),
        "stdev": statistics.stdev(times) if len(times) > 1 else 0.0,
        "min": min(times),
        "max": max(times),
    }

def print_table(label: str, text_s: dict, image_s: dict) -> None:
    """Print a formatted table comparing text and image execution times

    Args:
        label (str): The label for the table
        text_s (dict): A dictionary containing the mean, median, standard deviation, minimum, and maximum execution times for text processing in milliseconds
        image_s (dict): A dictionary containing the mean, median, standard deviation, minimum, and maximum execution times for image processing in milliseconds
    """

    print(f"\n{'─' * 57}")
    print(label)
    print(f"{'─' * 57}")
    print(f"{'':22} {'Text':>12} {'Image':>12}")
    for k, name in [("mean", "Average"), ("median", "Median"), ("stdev", "Std Dev"), ("min", "Minimum"), ("max", "Maximum")]:
        print(f"{name:22} {text_s[k]:>10.1f} ms {image_s[k]:>10.1f} ms")

## Text/image examples

In [4]:
TEST_TEXT = "brown fox jumps over the lazy dog"
TEST_IMAGE_URL = "https://4.img-dpreview.com/files/p/E~C667x0S5333x4000T1200x900~articles/3925134721/0266554465.jpeg"

response = requests.get(TEST_IMAGE_URL)
if response.status_code != 200:
    raise Exception(f"Failed to download image from {TEST_IMAGE_URL}, code: {response.status_code}")
TEST_IMAGE = Image.open(io.BytesIO(response.content)).convert("RGB")

## Benchmark params

In [5]:
WARMUP = 3
RUNS = 20

## Model metdata

In [6]:
MODEL_ID = "jinaai/jina-clip-v2"

## PyTorch timings

In [7]:
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()

with torch.no_grad():
    text_s = measure(
        lambda: model.encode_text([TEST_TEXT]),
        WARMUP,
        RUNS
    )
    image_s = measure(
        lambda: model.encode_image([TEST_IMAGE]),
        WARMUP,
        RUNS
    )
print_table("Original implementation", text_s, image_s)

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
/Users/alex/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_clip_hyphen_implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:137: UserWarning: Flash attention requires CUDA, disabling
  warnings.warn('Flash attention requires CUDA, disabling')
/Users/alex/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_clip_hyphen_implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:172: UserWarning: xFormers requires CUDA, disabling
  warnings.warn('xFormers requires CUDA, disabling')
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



─────────────────────────────────────────────────────────
Original implementation
─────────────────────────────────────────────────────────
                               Text        Image
Average                     555.7 ms     2052.6 ms
Median                      555.7 ms     2025.9 ms
Std Dev                       4.3 ms       60.8 ms
Minimum                     546.7 ms     2006.6 ms
Maximum                     565.2 ms     2222.6 ms


## ONNX timings

In [8]:
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.numpy()
    return np.asarray(x)

subfolder = "onnx"
model_filename = "model_int8.onnx"
local_path = hf_hub_download(
    repo_id=MODEL_ID,
    subfolder=subfolder,
    filename=model_filename,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

input_ids = to_numpy(tokenizer(TEST_TEXT, return_tensors='np', padding=True, truncation=True)['input_ids'])
pixel_values = to_numpy(image_processor(TEST_IMAGE, return_tensors='np')['pixel_values'])

options = ort.SessionOptions()
options.log_severity_level = 3
session = ort.InferenceSession(local_path, sess_options=options)

text_s = measure(
    lambda: session.run(None, {'input_ids': input_ids, 'pixel_values': np.zeros((0, 3, 512, 512), dtype=np.float32)}),
    WARMUP,
    RUNS
)
image_s = measure(
    lambda: session.run(None, {'input_ids': np.zeros((1, 1), dtype=np.int64), 'pixel_values': pixel_values}),
    WARMUP,
    RUNS
)
print_table("ONNX (int8)", text_s, image_s)


─────────────────────────────────────────────────────────
ONNX (int8)
─────────────────────────────────────────────────────────
                               Text        Image
Average                      14.5 ms     1558.0 ms
Median                       14.2 ms     1513.7 ms
Std Dev                       0.9 ms      140.7 ms
Minimum                      13.7 ms     1433.0 ms
Maximum                      16.7 ms     2076.2 ms


## Cosine similarity (text)

In [9]:
text_emb = model.encode_text([TEST_TEXT]).flatten()
text_emb_int8 = session.run(None, {'input_ids': input_ids, 'pixel_values': np.zeros((0, 3, 512, 512), dtype=np.float32)})[2].flatten()
np.dot(
    text_emb / np.linalg.norm(text_emb),
    text_emb_int8 / np.linalg.norm(text_emb_int8)
)

np.float32(0.7335259)

## Cosine similarity (image)

In [10]:
image_emb = model.encode_image([TEST_IMAGE]).flatten()
image_emb_int8 = session.run(None, {'input_ids': np.zeros((1, 1), dtype=np.int64), 'pixel_values': pixel_values})[3].flatten()
np.dot(
    image_emb / np.linalg.norm(image_emb),
    image_emb_int8 / np.linalg.norm(image_emb_int8)
)

np.float32(0.9847386)

In [11]:
print("--- ВХОДЫ МОДЕЛИ (INPUTS) ---")
for idx, node in enumerate(session.get_inputs()):
    print(f"Вход №{idx}:")
    print(f"  Имя (Name):   {node.name}")
    print(f"  Тип (Type):   {node.type}")
    print(f"  Форма (Shape): {node.shape}")
    print("-" * 30)

print("\n--- ВЫХОДЫ МОДЕЛИ (OUTPUTS) ---")
for idx, node in enumerate(session.get_outputs()):
    print(f"Выход №{idx}:")
    print(f"  Имя (Name):   {node.name}")
    print(f"  Тип (Type):   {node.type}")
    print(f"  Форма (Shape): {node.shape}")
    print("-" * 30)

--- ВХОДЫ МОДЕЛИ (INPUTS) ---
Вход №0:
  Имя (Name):   input_ids
  Тип (Type):   tensor(int64)
  Форма (Shape): ['batch_size', 'sequence_length']
------------------------------
Вход №1:
  Имя (Name):   pixel_values
  Тип (Type):   tensor(float)
  Форма (Shape): ['batch_size', 3, 512, 512]
------------------------------

--- ВЫХОДЫ МОДЕЛИ (OUTPUTS) ---
Выход №0:
  Имя (Name):   text_embeddings
  Тип (Type):   tensor(float)
  Форма (Shape): ['batch_size', 1024]
------------------------------
Выход №1:
  Имя (Name):   image_embeddings
  Тип (Type):   tensor(float)
  Форма (Shape): ['batch_size', 1024]
------------------------------
Выход №2:
  Имя (Name):   l2norm_text_embeddings
  Тип (Type):   tensor(float)
  Форма (Shape): ['batch_size', 1024]
------------------------------
Выход №3:
  Имя (Name):   l2norm_image_embeddings
  Тип (Type):   tensor(float)
  Форма (Shape): ['batch_size', 1024]
------------------------------
